# Wyszukiwarka


Celem projektu jest zaimplementowanie wyszukiwarki i zbadanie zastosowań Singular Value Decomposition w obszarze wyszukiwania informacji. Na końcu badam trafność zwróconych artykułów, w zależności od wykorzystanej metody wyszukiwania (klasycznego modelu wektorowego oraz modelu opartego na ukrytej semantyce SVD)


Główna wyszukiwarka działa dla 300_000 artykułów dla k=150

## 2. Struktura projektu
Projekt został podzielony na :
1) Moduł algorytmów i logiki matematycznej (`math_logic.py`)
- Budowanie słownika termów i rzadkiej macierzy (term-by-document matrix)
- Obliczanie i aplikowanie transformacji IDF
- Implementacja dekompozycji SVD oraz aproksymacji niskiego rzędu (LRA)
- Zmiana zapytań na wektory i obliczanie podobieństwa dla wyszukiwania klasycznego i semantycznego
2) Moduł przetwarzania tekstu( `text_preprocessing`)
- Normalizacja i tokenizacja tekstów
- usuwanie słów bez znaczenia (stopwords)
- implementacja stemmingu
3) Moduł pobierania i obsługi danych (`crawler.py`, `data_fetcher.py`)
- Asynchroniczne pobieranie artykułów z Wikipedii
- Zapisywanie tekstów do bazy SQLite
- Odpytywanie bazy o treści konkretnych dokumentów
4) Moduł konfiguracyjny i zapisu (`config.py`, `processed_data.py`)
- Konfiguracja głownych parametrów (współczynnik redukcji wielowymiarowości, limity pobierania dokumentów z bazy) oraz inicjalizacja NLTK
- Funkcje zapisujące i odczytujące przetworzone macierze, aby nie generować ich za każdym razem.
5) Serwer backendowy API (`app.py`)
6) Folder `frontend` , który odpowiada za interaktywną komunikacje z użytkownikiem czyli przyjmowanie zapytań i porównywanie wyników wyszukiwania obu modeli (TF-IDF vs SVD)


## Realizacja

### Budowa słownika i macierzy rzadkiej (term-by-document matrix)
Wstępne przygotowanie danych tekstowych oraz budowa macierzy zaimplementowałem w modułach `text_preprocessing.py` oraz `math_logic.py`
- Oczyszczanie tekstu (`clean_text`). Zastosowałem tokenizacje z zamianą wielkich liter na małe, odfiltrowałem znaki interpunkcyjne oraz usunięto stop-words (a, the itd.) za pomocą NLTK.
- Stemming. Skorzystałem z algorytmu PorterStemmer, która zmienia słowa do ich rdzeni (np. running -> run), co znacząco polepsza późniejsze grupowanie semantyczne.
- Konstrukcja macierzy (`build_word_document_matrix`). Zbudowałem słownik Counter, który zlicza mi ilość wystąpień danego termu w artykule. Odrzuciłem termy marginalne, które występują mniej niż 3 razy. To pozwala zredukować szum. Końcową macierz wygenerowałem w macierzy rzadkiej (`csr_matrix) z pakietu ScipPy.

## Transformacja IDF i normalizacja
Klasyczny model częstotliwościowy jest udoskolony w funkcji `apply_idf()`, która przekształca go w reprezentacje TF-IDF.
- Obliczanie wag z zabezpieczeniem: `IDF(term)=log(N/(n_w 1e-9))`. Dodanie stałej `1e-9` w mianowniku logarytmu zabezpiecza przed potencjalnym dzieleniem przez zero.
- Aplikacja wag: Wyliczone wartości IDF przemnożyłem przez pierwotną macierz częstotliwości, żeby nadać wyższą rangę rzadkim i specyficznym słowom, a zmniejszyć wpływ słow bardziej pospolitych, które nie dają nam wkładu kontekstu dokumentu.

## Dekompozycja SVD i redukcja szumu (LRA) (zrealizowane w funkcji `apply_svd_LRA()`)
- Obliczanie SVD: Do obliczeń wykorzystałem funkcje `svds` z modułu `scipy.sparse.linalg`, która pozwala na wyliczenie k największych wartości osobliwych macierzy.Wynik funkcji `svds` są macierze (U,S,V_T), które na razie są posortowane rosnąco.

## Klasyczne i semantyczne algorytmy wyszukiwania
Zaimplementowałem dwa osobne warianty wyszukiwarki:

- Metoda TF-IDF (`search_classic`): Zapytanie jest czyszczone (stemming), mapowane na odpowiednie indeksy w słowniku i mnożone przez wagi IDF. Znormalizowany wektor zapytania jest mnożony skalarnie przez znormalizowaną macierz TF-IDF.
- Metoda SVD (`search_semantic`): Przetworzony wektor zapytania nie jest porównywany bezpośrednio z dokumentami. Zamiast tego jest rzutowany do zredukowanej, k-wymiarowej przestrzeni przez pomnożenie przez transponowaną macierz `U_k`. Dopiero znormalizowany wektor zapytania w tej zredukowanej przestrzeni służy do wyliczenia ko wyliczenia korelacji ze skompresowaną macierzą dokumentów.

## Analiza działania
- wyszukiwanie z IDF i bez IDF
- wyszukiwania bez SVD i z SVD dla różnych wartości k (50,100,150,200,1000,2000) dla 30 000 artykułów (wyszukiwarka działa dla 300 000 artykułów, ale testy zajęły by dużo czasu)


In [18]:
import sys
import os
import time
import numpy as np
from sklearn.preprocessing import normalize
from IPython.display import Markdown, display

backend_path = os.path.join(os.getcwd(), 'backend')
if backend_path not in sys.path:
    sys.path.append(backend_path)

import backend.config
backend.config.DB_PATH = os.path.join(backend_path, 'data', 'wikipedia.db')

from backend.config import DB_PATH, init_resources
from backend.database.data_fetcher import fetch_articles_from_database
from backend.logic.math_logic import (
    build_word_document_matrix,
    apply_idf,
    apply_svd_LRA,
    search_classic,
    search_semantic
)

def main():
    init_resources()

    max_articles = 30000
    print(f"Status: Pobieranie {max_articles} artykułów z bazy danych...")
    articles = fetch_articles_from_database(DB_PATH, max_articles)

    print("Status: Budowa macierzy cech...")
    start_time = time.time()
    A, titles, word_to_idx = build_word_document_matrix(articles)
    matrix_time = round(time.time() - start_time, 2)

    print("Status: Obliczanie wag IDF...")
    A_tfidf, idf_values = apply_idf(A)

    print("Status: Normalizacja wektorów...")
    A_norm = normalize(A, axis=0, norm='l2')
    A_tfidf_norm = normalize(A_tfidf, axis=0, norm='l2')
    unit_idf_weights = np.ones(len(word_to_idx))

    queries = [
        "The first moon landing",
        "quantum physics",
        "history of Poland",
        "climate change",
        "donald trump election"
    ]

    k_values = [50, 100, 150, 200, 1000, 2000]
    results = {q: {} for q in queries}

    print("Status: Wykonywanie testów klasycznych i semantycznych...")
    for q in queries:
        results[q]["classic_idf"] = search_classic(q, word_to_idx, idf_values, A_tfidf_norm, titles, k=10)
        results[q]["classic_no_idf"] = search_classic(q, word_to_idx, unit_idf_weights, A_norm, titles, k=10)

    for k in k_values:
        u_idf, _, docs_idf = apply_svd_LRA(A_tfidf, k)
        norm_docs_idf = normalize(docs_idf, axis=0, norm='l2')
        u_no, _, docs_no = apply_svd_LRA(A, k)
        norm_docs_no = normalize(docs_no, axis=0, norm='l2')

        for q in queries:
            results[q][f"svd_{k}_idf"] = search_semantic(q, word_to_idx, idf_values, u_idf, norm_docs_idf, titles, k=10)
            results[q][f"svd_{k}_no_idf"] = search_semantic(q, word_to_idx, unit_idf_weights, u_no, norm_docs_no, titles, k=10)


    for q in queries:
        md_text = f"## Zapytanie: `{q}`\n\n"
        header = "| Konfiguracja | Klasyczne | " + " | ".join([f"SVD k={k}" for k in k_values]) + " |\n"
        sep = "| :--- | :--- | " + " | ".join([" :--- " for _ in k_values]) + " |\n"
        md_text += header + sep

        def format_res(res_list):
            if not res_list: return "Brak wyników"
            items = []
            for i, (t, s) in enumerate(res_list):
                short_t = t[:22] + "..." if len(t) > 25 else t
                items.append(f"{i+1}. {short_t} (**{s:.4f}**)")
            return "<br>".join(items)

        row1 = f"| **Z IDF** | {format_res(results[q]['classic_idf'])} | "
        row1 += " | ".join([format_res(results[q][f"svd_{k}_idf"]) for k in k_values]) + " |\n"

        row2 = f"| **Bez IDF** | {format_res(results[q]['classic_no_idf'])} | "
        row2 += " | ".join([format_res(results[q][f"svd_{k}_no_idf"]) for k in k_values]) + " |\n"

        md_text += row1 + row2
        display(Markdown(md_text))



In [19]:
main()

Status: Pobieranie 30000 artykułów z bazy danych...
Status: Budowa macierzy cech...
Status: Obliczanie wag IDF...
Status: Normalizacja wektorów...
Status: Wykonywanie testów klasycznych i semantycznych...


## Zapytanie: `The first moon landing`

| Konfiguracja | Klasyczne | SVD k=50 | SVD k=100 | SVD k=150 | SVD k=200 | SVD k=1000 | SVD k=2000 |
| :--- | :--- |  :---  |  :---  |  :---  |  :---  |  :---  |  :---  |
| **Z IDF** | 1. Sailor Moon (disambigu... (**0.5483**)<br>2. Richard Moon (**0.5016**)<br>3. Moon Patrol (**0.3738**)<br>4. Dewar (crater) (**0.3218**)<br>5. Destination Moon (album) (**0.3190**)<br>6. Illinois lunar sample ... (**0.2686**)<br>7. The Glimpses of the Mo... (**0.2583**)<br>8. Werewolf diet (**0.2191**)<br>9. Kob Kin Duean (**0.2080**)<br>10. Super Adventure Rockman (**0.2034**) | 1. Portage Lake (Alaska) (**0.8220**)<br>2. San Rafael (steamboat) (**0.8217**)<br>3. USS McKeever Brothers (**0.8158**)<br>4. Alaskan (1883 steamship) (**0.8119**)<br>5. Nathan Cobb Cottage (**0.8102**)<br>6. Burntcoat, Nova Scotia (**0.8100**)<br>7. Cranmore, Isle of Wight (**0.8082**)<br>8. French ship Foudroyant... (**0.8057**)<br>9. HNLMS De Ruyter (1853) (**0.8028**)<br>10. MV Mercedes I (**0.8000**) | 1. Burntcoat, Nova Scotia (**0.7294**)<br>2. Skipwith's Landing, Mi... (**0.7257**)<br>3. Carlsbergfondet Land (**0.7226**)<br>4. Point Barrow (**0.7089**)<br>5. Virgohamna (**0.7052**)<br>6. Tropical Storm Becky (... (**0.7037**)<br>7. Inninmore Bay (**0.7032**)<br>8. Coastline of Djursland (**0.6828**)<br>9. Hurricane Martha (**0.6766**)<br>10. Typhoon Kezia (**0.6718**) | 1. Carlsbergfondet Land (**0.6888**)<br>2. Rocket landing pad (**0.6648**)<br>3. American Land Conservancy (**0.6539**)<br>4. Inninmore Bay (**0.6417**)<br>5. Skipwith's Landing, Mi... (**0.6399**)<br>6. List of offshore wind ... (**0.6365**)<br>7. Burntcoat, Nova Scotia (**0.6268**)<br>8. Coastline of Djursland (**0.6129**)<br>9. Bleadon Hill (**0.6087**)<br>10. Point Barrow (**0.6055**) | 1. Rocket landing pad (**0.6671**)<br>2. Carlsbergfondet Land (**0.6110**)<br>3. Coastline of Djursland (**0.6046**)<br>4. Gianiskari beach (**0.5803**)<br>5. American Land Conservancy (**0.5737**)<br>6. Bleadon Hill (**0.5640**)<br>7. Skipwith's Landing, Mi... (**0.5566**)<br>8. Land footprint (**0.5563**)<br>9. Landing at Cape Helles (**0.5508**)<br>10. Inninmore Bay (**0.5342**) | 1. Sailor Moon (disambigu... (**0.6162**)<br>2. Dewar (crater) (**0.5024**)<br>3. Mesha Sankranti (**0.4668**)<br>4. Rocket landing pad (**0.4272**)<br>5. Khonsu (**0.4269**)<br>6. Werewolf diet (**0.4082**)<br>7. Kob Kin Duean (**0.3990**)<br>8. Franz von Paula Gruith... (**0.3958**)<br>9. Richard Moon (**0.3936**)<br>10. Solarquest (**0.3836**) | 1. Sailor Moon (disambigu... (**0.6260**)<br>2. Dewar (crater) (**0.5078**)<br>3. Werewolf diet (**0.4428**)<br>4. Mesha Sankranti (**0.4305**)<br>5. Richard Moon (**0.4289**)<br>6. Rocket landing pad (**0.4146**)<br>7. Land footprint (**0.3991**)<br>8. Moon Patrol (**0.3877**)<br>9. Lunarcrete (**0.3747**)<br>10. List of Artemis missions (**0.3645**) |
| **Bez IDF** | 1. Rocket landing pad (**0.3969**)<br>2. Sailor Moon (disambigu... (**0.3605**)<br>3. Land footprint (**0.3568**)<br>4. First Lady (disambigua... (**0.3553**)<br>5. List of first women la... (**0.3385**)<br>6. Rwanda Initiative for ... (**0.3282**)<br>7. American Land Conservancy (**0.3181**)<br>8. Korovou (**0.3123**)<br>9. Richard Moon (**0.3067**)<br>10. Skipwith's Landing, Mi... (**0.3013**) | 1. Ishimosorex (**0.8591**)<br>2. 1992 Palanca Awards (**0.8489**)<br>3. 1991 Palanca Awards (**0.8219**)<br>4. Tilicho Peak (**0.7850**)<br>5. First Lady (disambigua... (**0.7551**)<br>6. Coachella 2026 (**0.7242**)<br>7. Udren Zom (**0.7234**)<br>8. Luge at the 2018 Winte... (**0.7223**)<br>9. Gymnastics at the Euro... (**0.7104**)<br>10. Kaunik Acharya (**0.7085**) | 1. Ishimosorex (**0.8466**)<br>2. 1992 Palanca Awards (**0.7700**)<br>3. 1991 Palanca Awards (**0.7631**)<br>4. First Lady (disambigua... (**0.7478**)<br>5. Tilicho Peak (**0.7336**)<br>6. Andrzej Zawada (**0.6676**)<br>7. First Citizens Bank of... (**0.6627**)<br>8. Barbatia gens (**0.6559**)<br>9. Autosticha vicularis (**0.6559**)<br>10. PKP class SM40 (**0.6524**) | 1. Ishimosorex (**0.8080**)<br>2. First Lady (disambigua... (**0.7368**)<br>3. 1992 Palanca Awards (**0.7348**)<br>4. 1991 Palanca Awards (**0.7282**)<br>5. Tilicho Peak (**0.7188**)<br>6. Andrzej Zawada (**0.6790**)<br>7. List of first women la... (**0.6647**)<br>8. LGBTQ clergy in Judaism (**0.6567**)<br>9. Pelican Lake First Nation (**0.6540**)<br>10. Keystone First (**0.6476**) | 1. Ishimosorex (**0.7657**)<br>2. First Lady (disambigua... (**0.7281**)<br>3. Tilicho Peak (**0.6685**)<br>4. List of first women la... (**0.6514**)<br>5. 1992 Palanca Awards (**0.6508**)<br>6. Keystone First (**0.6497**)<br>7. Andrzej Zawada (**0.6490**)<br>8. 1991 Palanca Awards (**0.6431**)<br>9. LGBTQ clergy in Judaism (**0.6185**)<br>10. Sonnie Trotter (**0.6074**) | 1. Rocket landing pad (**0.5487**)<br>2. List of first women la... (**0.5365**)<br>3. Korovou (**0.5286**)<br>4. First Lady (disambigua... (**0.5267**)<br>5. Ishimosorex (**0.5109**)<br>6. Land footprint (**0.4858**)<br>7. Skipwith's Landing, Mi... (**0.4584**)<br>8. Rwanda Initiative for ... (**0.4543**)<br>9. Pelican Lake First Nation (**0.4409**)<br>10. Takeoff and landing (**0.4246**) | 1. Rocket landing pad (**0.5348**)<br>2. Land footprint (**0.4740**)<br>3. Korovou (**0.4611**)<br>4. Skipwith's Landing, Mi... (**0.4453**)<br>5. First Lady (disambigua... (**0.4361**)<br>6. Rwanda Initiative for ... (**0.4307**)<br>7. List of first women la... (**0.4254**)<br>8. Ishimosorex (**0.4081**)<br>9. Pelican Lake First Nation (**0.4010**)<br>10. Carlsbergfondet Land (**0.3955**) |


## Zapytanie: `quantum physics`

| Konfiguracja | Klasyczne | SVD k=50 | SVD k=100 | SVD k=150 | SVD k=200 | SVD k=1000 | SVD k=2000 |
| :--- | :--- |  :---  |  :---  |  :---  |  :---  |  :---  |  :---  |
| **Z IDF** | 1. Quantum Flagship (**0.6704**)<br>2. Benjamin Schumacher (**0.5602**)<br>3. Andris Ambainis (**0.4423**)<br>4. David Albert (**0.4124**)<br>5. Marvin Chester (**0.3490**)<br>6. Econophysics (**0.3136**)<br>7. James R. Chelikowsky (**0.3044**)<br>8. Chinese Physics C (**0.3027**)<br>9. Wu Ta-You (**0.3007**)<br>10. Paul Forman (historian) (**0.2794**) | 1. Systems neuroscience (**0.8507**)<br>2. Chinese Physics C (**0.8350**)<br>3. SMG1 (**0.8306**)<br>4. S-phase kinase-associa... (**0.8266**)<br>5. Psychophysiological in... (**0.8249**)<br>6. CDH8 (**0.8126**)<br>7. Christopher Sachrajda (**0.8030**)<br>8. MiR-134 (**0.7968**)<br>9. PPP1CA (**0.7934**)<br>10. Theta wave (**0.7918**) | 1. Photon (**0.9000**)<br>2. Cavity optomechanics (**0.8559**)<br>3. Anapole (**0.8556**)<br>4. Lepton (**0.8448**)<br>5. PeVatron (**0.8352**)<br>6. Benjamin Schumacher (**0.8265**)<br>7. Christopher Sachrajda (**0.8255**)<br>8. Two-photon photoelectr... (**0.8227**)<br>9. Plasmonics (**0.7853**)<br>10. Centre for Device Ther... (**0.7735**) | 1. Photon (**0.9527**)<br>2. Anapole (**0.9229**)<br>3. PeVatron (**0.8875**)<br>4. Benjamin Schumacher (**0.8660**)<br>5. Cavity optomechanics (**0.8381**)<br>6. Gamma camera (**0.8120**)<br>7. Andris Ambainis (**0.8018**)<br>8. Xi-Cheng Zhang (**0.7706**)<br>9. Lepton (**0.7702**)<br>10. Christopher Sachrajda (**0.7609**) | 1. Photon (**0.9388**)<br>2. Anapole (**0.9097**)<br>3. Benjamin Schumacher (**0.8424**)<br>4. PeVatron (**0.8089**)<br>5. Gamma camera (**0.7993**)<br>6. Cavity optomechanics (**0.7938**)<br>7. Andris Ambainis (**0.7806**)<br>8. Xi-Cheng Zhang (**0.7493**)<br>9. Christopher Sachrajda (**0.7175**)<br>10. Two-photon photoelectr... (**0.7151**) | 1. Benjamin Schumacher (**0.8289**)<br>2. Marvin Chester (**0.6255**)<br>3. David Albert (**0.5980**)<br>4. Physical therapy (**0.5947**)<br>5. Jacques Herbrand Prize (**0.5937**)<br>6. Juan Maldacena (**0.5927**)<br>7. Christopher Sachrajda (**0.5898**)<br>8. Andris Ambainis (**0.5887**)<br>9. Chinese Physics C (**0.5347**)<br>10. Arkady Migdal (**0.5321**) | 1. Benjamin Schumacher (**0.8149**)<br>2. David Albert (**0.6161**)<br>3. Econophysics (**0.6058**)<br>4. Marvin Chester (**0.5942**)<br>5. Juan Maldacena (**0.5634**)<br>6. Christopher Sachrajda (**0.5498**)<br>7. Quantum Flagship (**0.5474**)<br>8. Andris Ambainis (**0.5433**)<br>9. Arkady Migdal (**0.5392**)<br>10. Chinese Physics C (**0.5103**) |
| **Bez IDF** | 1. Benjamin Schumacher (**0.5539**)<br>2. Chinese Physics C (**0.5045**)<br>3. Physical therapy (**0.4436**)<br>4. Jacques Herbrand Prize (**0.4262**)<br>5. Quantum Flagship (**0.3827**)<br>6. Wu Ta-You (**0.3674**)<br>7. Paul Forman (historian) (**0.3575**)<br>8. Heather Lewandowski (**0.3412**)<br>9. Jon Butterworth (**0.3408**)<br>10. Joan Warnow-Blewett (**0.3351**) | 1. End-of-life ethics (**0.8040**)<br>2. Comprehensive medicati... (**0.8004**)<br>3. Physical therapy (**0.7935**)<br>4. Mental health (**0.7755**)<br>5. Relationships and health (**0.7663**)<br>6. Health ecology (**0.7521**)<br>7. Polypharmacy (**0.7506**)<br>8. Health (Apple) (**0.7458**)<br>9. Black maternal mortali... (**0.7456**)<br>10. Self-diagnosis (**0.7429**) | 1. Physical therapy (**0.6394**)<br>2. At risk mental state (**0.5708**)<br>3. Relationships and health (**0.5674**)<br>4. John Wayne Mason (**0.5553**)<br>5. Systems neuroscience (**0.5452**)<br>6. End-of-life ethics (**0.5446**)<br>7. Bioenergy economy-base... (**0.5442**)<br>8. Sleep medicine (**0.5310**)<br>9. Association for Behavi... (**0.5302**)<br>10. Tbx18 transduction (**0.5291**) | 1. Photon (**0.6890**)<br>2. Double layer (plasma p... (**0.5816**)<br>3. Speed of gravity (**0.5673**)<br>4. Lepton (**0.5621**)<br>5. Waxman-Bahcall bound (**0.5509**)<br>6. Benjamin Schumacher (**0.5373**)<br>7. Physical therapy (**0.5325**)<br>8. Cavity optomechanics (**0.5319**)<br>9. PeVatron (**0.5267**)<br>10. Two-photon photoelectr... (**0.5155**) | 1. Photon (**0.7367**)<br>2. Speed of gravity (**0.5999**)<br>3. Benjamin Schumacher (**0.5986**)<br>4. Waxman-Bahcall bound (**0.5772**)<br>5. Lepton (**0.5719**)<br>6. Magnetism (**0.5556**)<br>7. Physical therapy (**0.5533**)<br>8. Double layer (plasma p... (**0.5516**)<br>9. Chinese Physics C (**0.5436**)<br>10. PeVatron (**0.5347**) | 1. Physical therapy (**0.7593**)<br>2. Jacques Herbrand Prize (**0.7175**)<br>3. Chinese Physics C (**0.7026**)<br>4. Benjamin Schumacher (**0.5900**)<br>5. Francis F. Chen (**0.5774**)<br>6. Jon Butterworth (**0.5714**)<br>7. Juan Maldacena (**0.5181**)<br>8. Christopher Sachrajda (**0.5016**)<br>9. Matthieu Wyart (**0.4968**)<br>10. Joan Warnow-Blewett (**0.4835**) | 1. Chinese Physics C (**0.6828**)<br>2. Physical therapy (**0.6496**)<br>3. Jacques Herbrand Prize (**0.6303**)<br>4. Benjamin Schumacher (**0.5899**)<br>5. Jon Butterworth (**0.5511**)<br>6. Francis F. Chen (**0.5500**)<br>7. Juan Maldacena (**0.4910**)<br>8. Wu Ta-You (**0.4755**)<br>9. Matthieu Wyart (**0.4735**)<br>10. Joan Warnow-Blewett (**0.4707**) |


## Zapytanie: `history of Poland`

| Konfiguracja | Klasyczne | SVD k=50 | SVD k=100 | SVD k=150 | SVD k=200 | SVD k=1000 | SVD k=2000 |
| :--- | :--- |  :---  |  :---  |  :---  |  :---  |  :---  |  :---  |
| **Z IDF** | 1. Ostrogóra (**0.3876**)<br>2. Kolonia Osiek (**0.3826**)<br>3. Stefanków, Szydłowiec ... (**0.3642**)<br>4. Aleksander Kakowski (**0.3320**)<br>5. 2022 FIFA World Cup Gr... (**0.3274**)<br>6. Żołędowo, Kuyavian-Pom... (**0.3247**)<br>7. Poland (novel) (**0.2840**)<br>8. Celiny (**0.2831**)<br>9. Sokołowo, Września County (**0.2776**)<br>10. Zofiówka (**0.2721**) | 1. List of massacres of B... (**0.8510**)<br>2. Deutscher Historikertag (**0.8377**)<br>3. Joseph Gébara (**0.8352**)<br>4. Christi Matri (**0.8349**)<br>5. Paul Mojzes (**0.8308**)<br>6. Political views on the... (**0.8306**)<br>7. Catholic Church in Vie... (**0.8290**)<br>8. Joannicius III of Cons... (**0.8199**)<br>9. Uniting Reformed Churc... (**0.8191**)<br>10. Mieczysław Jastrun (**0.8164**) | 1. Waldstein (**0.7204**)<br>2. Mieczysław Jastrun (**0.7190**)<br>3. Carl von Weinberg (**0.6874**)<br>4. Deutscher Historikertag (**0.6839**)<br>5. Inowrocław (**0.6825**)<br>6. August von Wörndle (**0.6816**)<br>7. Sophie Rogge-Börner (**0.6793**)<br>8. Ernest Bramsted (**0.6731**)<br>9. Wolfgang Willrich (**0.6731**)<br>10. Kowary (**0.6723**) | 1. Jagiellonian dynasty (**0.7168**)<br>2. Willa of Tuscany (**0.6667**)<br>3. Piltene (**0.6619**)<br>4. Kowary (**0.6607**)<br>5. Prozor Fortress (**0.6541**)<br>6. Krzysztof Radziwiłł (**0.6377**)<br>7. List of Lithuanian roy... (**0.6362**)<br>8. Francis II (**0.6285**)<br>9. Dabiša of Bosnia (**0.6280**)<br>10. Angelina (surname) (**0.6240**) | 1. Jagiellonian dynasty (**0.7757**)<br>2. Piltene (**0.7159**)<br>3. List of Lithuanian roy... (**0.6953**)<br>4. Krzysztof Radziwiłł (**0.6639**)<br>5. Chronica Hungarorum (**0.6165**)<br>6. Dabiša of Bosnia (**0.6095**)<br>7. Kowary (**0.6075**)<br>8. Sokołowo, Września County (**0.5920**)<br>9. Mieczysław Jastrun (**0.5895**)<br>10. Hermann Billung (**0.5871**) | 1. Celiny (**0.8478**)<br>2. Jagiellonian dynasty (**0.8067**)<br>3. Lasek (**0.7426**)<br>4. Sokołowo, Września County (**0.7285**)<br>5. Stefanków, Szydłowiec ... (**0.7212**)<br>6. Borowina (**0.7190**)<br>7. Zofiówka (**0.7043**)<br>8. Henryk Dembiński (**0.6671**)<br>9. Oszyk coat of arms (**0.6634**)<br>10. Kot (**0.6539**) | 1. Celiny (**0.8574**)<br>2. Lasek (**0.7931**)<br>3. Borowina (**0.7719**)<br>4. Zofiówka (**0.7628**)<br>5. Stefanków, Szydłowiec ... (**0.6880**)<br>6. Sokołowo, Września County (**0.6626**)<br>7. Jagiellonian dynasty (**0.6432**)<br>8. Aleksander Kakowski (**0.6282**)<br>9. Poland (novel) (**0.6136**)<br>10. Ostrogóra (**0.6017**) |
| **Bez IDF** | 1. Judith M. Bennett (**0.3939**)<br>2. Anuario de Historia de... (**0.3795**)<br>3. Celiny (**0.3776**)<br>4. Zofiówka (**0.3632**)<br>5. Martyn Lyons (**0.3433**)<br>6. Borowina (**0.3421**)<br>7. Stefanków, Szydłowiec ... (**0.3310**)<br>8. Carl N. Degler (**0.3276**)<br>9. Kolonia Osiek (**0.3244**)<br>10. Andreas Gestrich (**0.3194**) | 1. Czech National Library... (**0.7406**)<br>2. How the World Made the... (**0.7044**)<br>3. National Museum of Sci... (**0.6469**)<br>4. Outline of technology (**0.6466**)<br>5. Martin S. Peterson (**0.6423**)<br>6. Rambabu Kodali (**0.6315**)<br>7. Pauline van Dongen (**0.6238**)<br>8. Kastanoussa (**0.6222**)<br>9. Ithiel de Sola Pool (**0.6196**)<br>10. Technology Professiona... (**0.6164**) | 1. Kastanoussa (**0.5840**)<br>2. Czech National Library... (**0.5457**)<br>3. Elizabeth Redgate (**0.5269**)<br>4. Sripurusha (**0.5268**)<br>5. How the World Made the... (**0.5263**)<br>6. Diləli Müskənli (**0.5178**)<br>7. Etruscan history (**0.5157**)<br>8. Zoğalbulaq (**0.5145**)<br>9. NZZ Geschichte (**0.5126**)<br>10. Qaracallı, Qubadli (**0.5084**) | 1. Seodaemun Museum of Na... (**0.5647**)<br>2. West Tennessee Histori... (**0.5126**)<br>3. John and Annie Glenn M... (**0.5117**)<br>4. Anuario de Historia de... (**0.5103**)<br>5. Queens Historical Society (**0.5101**)<br>6. The Medieval History J... (**0.5075**)<br>7. Václavovice (**0.4984**)<br>8. Maryanne Cline Horowitz (**0.4979**)<br>9. NZZ Geschichte (**0.4961**)<br>10. Franklin Garrett (**0.4952**) | 1. Seodaemun Museum of Na... (**0.5761**)<br>2. Maryanne Cline Horowitz (**0.5331**)<br>3. Anuario de Historia de... (**0.5279**)<br>4. Queens Historical Society (**0.5232**)<br>5. History of Maine (**0.5204**)<br>6. Eliza Atkins Gleason B... (**0.5127**)<br>7. Martyn Lyons (**0.4900**)<br>8. Franklin Garrett (**0.4885**)<br>9. The History of Rome (p... (**0.4826**)<br>10. The Medieval History J... (**0.4764**) | 1. Anuario de Historia de... (**0.6217**)<br>2. Judith M. Bennett (**0.6060**)<br>3. Martyn Lyons (**0.5977**)<br>4. Andreas Gestrich (**0.5217**)<br>5. Carl N. Degler (**0.4999**)<br>6. Orunia Górna-Gdańsk Po... (**0.4952**)<br>7. Maryanne Cline Horowitz (**0.4913**)<br>8. Erwin Ackerknecht (**0.4837**)<br>9. David Brody (historian) (**0.4813**)<br>10. Dean C. Allard (**0.4752**) | 1. Judith M. Bennett (**0.5690**)<br>2. Anuario de Historia de... (**0.5535**)<br>3. Andreas Gestrich (**0.4813**)<br>4. Carl N. Degler (**0.4690**)<br>5. Maryanne Cline Horowitz (**0.4540**)<br>6. Martyn Lyons (**0.4522**)<br>7. Orunia Górna-Gdańsk Po... (**0.4486**)<br>8. Erwin Ackerknecht (**0.4470**)<br>9. Dean C. Allard (**0.4343**)<br>10. David Brody (historian) (**0.4328**) |


## Zapytanie: `climate change`

| Konfiguracja | Klasyczne | SVD k=50 | SVD k=100 | SVD k=150 | SVD k=200 | SVD k=1000 | SVD k=2000 |
| :--- | :--- |  :---  |  :---  |  :---  |  :---  |  :---  |  :---  |
| **Z IDF** | 1. Climate crisis (**0.7520**)<br>2. Climate Rush (**0.6753**)<br>3. Effects of climate change (**0.6529**)<br>4. Tropical rainforest cl... (**0.5530**)<br>5. Climate Science Rapid ... (**0.5503**)<br>6. Freney (**0.5363**)<br>7. Richards, Texas (**0.3939**)<br>8. Emissions Trading Sche... (**0.3833**)<br>9. Harchin (**0.3685**)<br>10. Richard Alley (**0.3637**) | 1. Tropical rainforest cl... (**0.9340**)<br>2. Effects of climate change (**0.9209**)<br>3. Massoia's lutrine opossum (**0.9114**)<br>4. Beaufortia leveretti (**0.9090**)<br>5. Philodendron pedatum (**0.9064**)<br>6. Craugastor laevissimus (**0.8969**)<br>7. Acropora nasuta (**0.8933**)<br>8. Gerbe's vole (**0.8899**)<br>9. Sierra Nevada subalpin... (**0.8881**)<br>10. Ctenobrycon (**0.8863**) | 1. Tropical rainforest cl... (**0.9292**)<br>2. Effects of climate change (**0.8969**)<br>3. Shira, Nigeria (**0.8635**)<br>4. Freney (**0.8588**)<br>5. Climate change in Nunavut (**0.8382**)<br>6. Sierra Nevada subalpin... (**0.8354**)<br>7. Cambisol (**0.8353**)<br>8. Sandamu (**0.8326**)<br>9. Climate crisis (**0.8258**)<br>10. Seaweed fertiliser (**0.8223**) | 1. Tropical rainforest cl... (**0.9489**)<br>2. Effects of climate change (**0.9011**)<br>3. Freney (**0.8898**)<br>4. Climate crisis (**0.8746**)<br>5. Shira, Nigeria (**0.8693**)<br>6. Sandamu (**0.8337**)<br>7. Climate change in Nunavut (**0.8203**)<br>8. Gifford, Washington (**0.8097**)<br>9. Harchin (**0.8072**)<br>10. Paleocene (**0.7968**) | 1. Tropical rainforest cl... (**0.9520**)<br>2. Freney (**0.9325**)<br>3. Climate crisis (**0.9263**)<br>4. Effects of climate change (**0.9161**)<br>5. Harchin (**0.8784**)<br>6. Shira, Nigeria (**0.8769**)<br>7. Gifford, Washington (**0.8758**)<br>8. Climate Rush (**0.8707**)<br>9. Sandamu (**0.8536**)<br>10. Paleocene (**0.8352**) | 1. Climate crisis (**0.9624**)<br>2. Freney (**0.9202**)<br>3. Climate Rush (**0.9196**)<br>4. Climate Science Rapid ... (**0.8732**)<br>5. Effects of climate change (**0.8730**)<br>6. Gifford, Washington (**0.8552**)<br>7. Sophia Kianni (**0.8504**)<br>8. Tropical rainforest cl... (**0.8504**)<br>9. Harchin (**0.8399**)<br>10. Climate change in Nunavut (**0.7868**) | 1. Climate crisis (**0.9283**)<br>2. Climate Rush (**0.8812**)<br>3. Freney (**0.8607**)<br>4. Climate Science Rapid ... (**0.8243**)<br>5. Tropical rainforest cl... (**0.8081**)<br>6. Effects of climate change (**0.7808**)<br>7. Harchin (**0.7806**)<br>8. Gifford, Washington (**0.7773**)<br>9. Sanders, Montana (**0.7113**)<br>10. Climate change in Nunavut (**0.7010**) |
| **Bez IDF** | 1. Climate crisis (**0.6905**)<br>2. Effects of climate change (**0.6859**)<br>3. Climate Rush (**0.6510**)<br>4. Tropical rainforest cl... (**0.5169**)<br>5. Climate change in Nunavut (**0.4642**)<br>6. Sophia Kianni (**0.4408**)<br>7. Richard Alley (**0.4360**)<br>8. Freney (**0.4111**)<br>9. Emissions Trading Sche... (**0.4002**)<br>10. Climate Science Rapid ... (**0.3895**) | 1. Effects of climate change (**0.8534**)<br>2. Tropical rainforest cl... (**0.8083**)<br>3. Sierra Nevada subalpin... (**0.7521**)<br>4. Climate change in Nunavut (**0.7405**)<br>5. List of maximum animal... (**0.7309**)<br>6. Climate crisis (**0.7258**)<br>7. Brown planthopper (**0.7256**)<br>8. Channa punctata (**0.7245**)<br>9. Human evolution (**0.7213**)<br>10. Gerbe's vole (**0.7191**) | 1. Effects of climate change (**0.8473**)<br>2. Tropical rainforest cl... (**0.8382**)<br>3. Climate crisis (**0.7864**)<br>4. Climate change in Nunavut (**0.7061**)<br>5. Fuse plug (**0.6836**)<br>6. Bouguer (Martian crater) (**0.6702**)<br>7. Freney (**0.6588**)<br>8. Water supply and sanit... (**0.6545**)<br>9. Climate Rush (**0.6538**)<br>10. Copernicus (Martian cr... (**0.6534**) | 1. Tropical rainforest cl... (**0.8949**)<br>2. Climate crisis (**0.8945**)<br>3. Effects of climate change (**0.8862**)<br>4. Climate Rush (**0.8429**)<br>5. Climate change in Nunavut (**0.7521**)<br>6. Freney (**0.7480**)<br>7. Harchin (**0.6749**)<br>8. Sandamu (**0.6382**)<br>9. Gifford, Washington (**0.6231**)<br>10. Shira, Nigeria (**0.6202**) | 1. Climate crisis (**0.9176**)<br>2. Effects of climate change (**0.8825**)<br>3. Tropical rainforest cl... (**0.8758**)<br>4. Climate Rush (**0.8741**)<br>5. Climate change in Nunavut (**0.7808**)<br>6. Freney (**0.7581**)<br>7. Harchin (**0.6864**)<br>8. Gifford, Washington (**0.6685**)<br>9. Climate Science Rapid ... (**0.6484**)<br>10. Richard Alley (**0.6450**) | 1. Climate crisis (**0.7973**)<br>2. Effects of climate change (**0.7873**)<br>3. Climate Rush (**0.7783**)<br>4. Tropical rainforest cl... (**0.6980**)<br>5. Climate change in Nunavut (**0.6391**)<br>6. Sophia Kianni (**0.5677**)<br>7. Richard Alley (**0.5464**)<br>8. Harchin (**0.5320**)<br>9. Freney (**0.5273**)<br>10. Gifford, Washington (**0.5125**) | 1. Climate crisis (**0.7506**)<br>2. Effects of climate change (**0.7379**)<br>3. Climate Rush (**0.7164**)<br>4. Tropical rainforest cl... (**0.6289**)<br>5. Climate change in Nunavut (**0.5795**)<br>6. Sophia Kianni (**0.5187**)<br>7. Richard Alley (**0.5154**)<br>8. Freney (**0.4679**)<br>9. Deliberatorium (**0.4608**)<br>10. Gifford, Washington (**0.4580**) |


## Zapytanie: `donald trump election`

| Konfiguracja | Klasyczne | SVD k=50 | SVD k=100 | SVD k=150 | SVD k=200 | SVD k=1000 | SVD k=2000 |
| :--- | :--- |  :---  |  :---  |  :---  |  :---  |  :---  |  :---  |
| **Z IDF** | 1. Personal and business ... (**0.7309**)<br>2. Trump v. Internal Reve... (**0.5842**)<br>3. Byron Donalds (**0.5016**)<br>4. Stormy Daniels–Donald ... (**0.4868**)<br>5. Executive Office appoi... (**0.4502**)<br>6. Angela Stanton-King (**0.3478**)<br>7. American Oversight (**0.3270**)<br>8. World Liberty Financial (**0.3260**)<br>9. Fred Spencer (**0.2925**)<br>10. Al Green (politician) (**0.2740**) | 1. Personal and business ... (**0.9907**)<br>2. Stormy Daniels–Donald ... (**0.9865**)<br>3. Trump v. Internal Reve... (**0.9777**)<br>4. World Liberty Financial (**0.9585**)<br>5. American Oversight (**0.9252**)<br>6. Trump squeeze (**0.9223**)<br>7. Executive Office appoi... (**0.9134**)<br>8. Armand Cucciniello (**0.9127**)<br>9. Jim Donovan (banker) (**0.8853**)<br>10. Elmar Theveßen (**0.8451**) | 1. Personal and business ... (**0.9841**)<br>2. Stormy Daniels–Donald ... (**0.9775**)<br>3. Trump v. Internal Reve... (**0.9717**)<br>4. World Liberty Financial (**0.9341**)<br>5. American Oversight (**0.9095**)<br>6. Trump squeeze (**0.9079**)<br>7. Executive Office appoi... (**0.8922**)<br>8. Armand Cucciniello (**0.8685**)<br>9. Jim Donovan (banker) (**0.8314**)<br>10. 44th G7 summit (**0.8010**) | 1. Personal and business ... (**0.9729**)<br>2. Stormy Daniels–Donald ... (**0.9598**)<br>3. Trump v. Internal Reve... (**0.9511**)<br>4. World Liberty Financial (**0.9146**)<br>5. American Oversight (**0.8893**)<br>6. Trump squeeze (**0.8790**)<br>7. Executive Office appoi... (**0.8744**)<br>8. Armand Cucciniello (**0.8570**)<br>9. Jim Donovan (banker) (**0.7950**)<br>10. Angela Stanton-King (**0.7838**) | 1. Personal and business ... (**0.9647**)<br>2. Trump v. Internal Reve... (**0.9409**)<br>3. World Liberty Financial (**0.9063**)<br>4. American Oversight (**0.8791**)<br>5. Executive Office appoi... (**0.8716**)<br>6. Trump squeeze (**0.8512**)<br>7. Armand Cucciniello (**0.8491**)<br>8. Jim Donovan (banker) (**0.7886**)<br>9. Angela Stanton-King (**0.7643**)<br>10. 44th G7 summit (**0.7528**) | 1. Personal and business ... (**0.9242**)<br>2. Trump v. Internal Reve... (**0.8852**)<br>3. World Liberty Financial (**0.7765**)<br>4. Executive Office appoi... (**0.7714**)<br>5. American Oversight (**0.7650**)<br>6. Armand Cucciniello (**0.7410**)<br>7. Trump squeeze (**0.6988**)<br>8. Angela Stanton-King (**0.6391**)<br>9. Stormy Daniels–Donald ... (**0.6151**)<br>10. 44th G7 summit (**0.5834**) | 1. Personal and business ... (**0.9000**)<br>2. Trump v. Internal Reve... (**0.8391**)<br>3. American Oversight (**0.6917**)<br>4. Executive Office appoi... (**0.6876**)<br>5. World Liberty Financial (**0.6751**)<br>6. Armand Cucciniello (**0.6708**)<br>7. Stormy Daniels–Donald ... (**0.5988**)<br>8. Angela Stanton-King (**0.5731**)<br>9. Trump squeeze (**0.5468**)<br>10. 44th G7 summit (**0.4421**) |
| **Bez IDF** | 1. Personal and business ... (**0.5613**)<br>2. Electoral results for ... (**0.5025**)<br>3. Electoral results for ... (**0.4767**)<br>4. Byron Donalds (**0.4762**)<br>5. 2024 elections in Bang... (**0.4179**)<br>6. 2026 elections in the ... (**0.4018**)<br>7. Saptari 4 (**0.3998**)<br>8. 1960 elections in India (**0.3984**)<br>9. Forfarshire (UK Parlia... (**0.3961**)<br>10. Redbridge (electoral d... (**0.3961**) | 1. Personal and business ... (**0.8404**)<br>2. Stormy Daniels–Donald ... (**0.8337**)<br>3. Trump v. Internal Reve... (**0.7886**)<br>4. Executive Office appoi... (**0.6603**)<br>5. Al Green (politician) (**0.6181**)<br>6. Angela Stanton-King (**0.6143**)<br>7. World Liberty Financial (**0.5935**)<br>8. Post-presidency of Geo... (**0.5670**)<br>9. American Oversight (**0.5557**)<br>10. Gennady Zyuganov 2008 ... (**0.5526**) | 1. Personal and business ... (**0.7875**)<br>2. Stormy Daniels–Donald ... (**0.7688**)<br>3. Trump v. Internal Reve... (**0.7038**)<br>4. Electoral results for ... (**0.5816**)<br>5. Electoral results for ... (**0.5751**)<br>6. Executive Office appoi... (**0.5715**)<br>7. Haveli, Maharashtra As... (**0.5647**)<br>8. Hunasuru Assembly cons... (**0.5641**)<br>9. List of elections in 1925 (**0.5632**)<br>10. Saptari 4 (**0.5620**) | 1. Personal and business ... (**0.7656**)<br>2. Stormy Daniels–Donald ... (**0.7207**)<br>3. Trump v. Internal Reve... (**0.6517**)<br>4. Electoral results for ... (**0.6011**)<br>5. Electoral results for ... (**0.6006**)<br>6. List of elections in 1925 (**0.5932**)<br>7. Haveli, Maharashtra As... (**0.5902**)<br>8. Saptari 4 (**0.5875**)<br>9. Hunasuru Assembly cons... (**0.5871**)<br>10. Redbridge (electoral d... (**0.5833**) | 1. Personal and business ... (**0.7562**)<br>2. Stormy Daniels–Donald ... (**0.6468**)<br>3. Trump v. Internal Reve... (**0.6321**)<br>4. Electoral results for ... (**0.5993**)<br>5. Electoral results for ... (**0.5976**)<br>6. List of elections in 1925 (**0.5892**)<br>7. Saptari 4 (**0.5847**)<br>8. List of elections in 1953 (**0.5814**)<br>9. Redbridge (electoral d... (**0.5806**)<br>10. Hackney (electoral div... (**0.5802**) | 1. Personal and business ... (**0.6797**)<br>2. Electoral results for ... (**0.6167**)<br>3. List of elections in 1925 (**0.6118**)<br>4. Electoral results for ... (**0.6048**)<br>5. List of elections in 1953 (**0.5973**)<br>6. Saptari 4 (**0.5593**)<br>7. Lewisham (electoral di... (**0.5568**)<br>8. Redbridge (electoral d... (**0.5525**)<br>9. Hackney (electoral div... (**0.5517**)<br>10. 2024 elections in Bang... (**0.5459**) | 1. Personal and business ... (**0.6514**)<br>2. Electoral results for ... (**0.5951**)<br>3. List of elections in 1925 (**0.5870**)<br>4. Electoral results for ... (**0.5756**)<br>5. List of elections in 1953 (**0.5473**)<br>6. Saptari 4 (**0.5141**)<br>7. 2024 elections in Bang... (**0.5138**)<br>8. 2026 elections in the ... (**0.5137**)<br>9. Redbridge (electoral d... (**0.5120**)<br>10. Lewisham (electoral di... (**0.5117**) |


## Wnioski

### Wpływ IDF
W przeprowadzonym eksperymencie widać, że użycie transpormacji IDF jest waże dla jakości wyników. Bez wag IDF algorytm faworyzuje słowa pospolite. Na przykład, przy zapytaniu `donald trump election`, model `bez IDF` opierał się głównie na słowie `election`, zwracając artykuły nie koniecznie o wyborach Donalda Trumpa, lecz o losowych wyborach (np. w Bangladeszu czy Indiach). IDF promuje specyficzne dla zapytania słowa (termy).

### Wpływ SVD
- Wyniki z SVD pokazują, że algorytm znajduje dokumenty powiązane znaczeniowo, nawet bez dosłownego dopasowania termów. Dobrym przykładem jest zapytanie `history of Poland`, dla którego SVD na wysokiej pozycji znalazło `Jagiellonian dynasty`.
- Zastosowanie SVD pozwala również na lepsze zrozumienie kontekstu specjalistycznego. Na przykład dla zapytania `quantum physics` , w przeciwieństwie do klasycznego modelu, model semantyczny znalazł powiązane pojęcia jak `Photon`, `Lepton`.
- Gdy analizujemy małą liczbę dokumentów (30 000), model jest podatny na nadreprezentacje pewnych słów. Przy zapytaniu `The first moon landing` algorytm powiązał je z portami morskimi (`Skipwith's Landing`). Dzieje się to prawdopodobnie dlatego że słowo `landing` częściej występuje w kontekście portowym.
### Wplyw wartości k w SVD
- Niższe wartości k=50: Widać że zbyt generalizuje tematy (za bardzo kompresuje macierz). Prowadzi to do zlewania się odrębnych pojęć (np. dla fizyki kwantowej algorytm zaczął zwracać artykuły o neurobiologi systemowej).
- Wartości k (100-200). Wydaje się, że to wartości optymalne. Model poprawnie dopasowuje powiązania znaczeniowe, odrzucając w międzyczasie szum.
- Wyższe wartości k>1000: Wyniki przypominają z powrotem klasyczną metodę TF-IDF. Oznacza to, że przybliżenie staje się zbyt dokładne, gdzie szum staje się znów widoczny. Czyli tracimy właściwości SVD, a dodatkowo zwiększa się koszt obliczeniowy)

W głównej wyszukiwarce wykorzystuje 300 000 artykułów z bazy danych, gdzie k=300. W sprawozdaniu testowałem dla 30 000 artykułów, ze względu na ograniczenia sprzętowe.

# Ogólne wnioski

- Klasyczne wyszukiwanie (TF-IDF): Model stosuje wyszukiwanie dosłowne. Ma ograniczoną zdolność do znajdowania semantycznie podobnych artykułów i nie potrafi powiązać synonimów ze sobą.
- Wyszukiwanie z SVD: Potrafi znajdować głębokie powiązania pojęciowe ( co widać było w temacie fizyki). Jednak dzieje się to kosztem dużej złożoności obliczeniowej i ryzykiem błędnych powiązań przy źle dobranym parametrze k.

